In [1]:
DATA_DIR= "" 
SEP = ";"

In [2]:
import pandas as pd

df = pd.read_csv('parametros_globales.csv', delimiter=';')
df['parametro'] = df['parametro'].str.strip()
params = ['I', 'T', 'P', 'C', 'K', 'M', 'S', 'CI']
valores = df.set_index('parametro')['valor'].reindex(params).tolist()
I, T, P, C, K, M, S, CI = valores
print(f'I: {I}, T: {T}, P: {P}, C: {C}, K: {K}, M: {M}, S: {S}, CI: {CI}')

I: 1500, T: 18, P: 13, C: 26, K: 23, M: 4, S: 19, CI: 24


In [3]:
import numpy as np
# Limpieza pabellones
L = {
        p: int(np.random.randint(1, 4))  # 1 a 3 bloques
        for p in range(1, P + 1)
    }

df_limpieza = pd.DataFrame({
        "p": list(range(1, P + 1)),
        "L": [L[p] for p in range(1, P + 1)],
    })

df_limpieza.to_csv(DATA_DIR + "limpieza_pabellones.csv", index=False, sep=SEP)


In [4]:
# Prioridad pacientes
mu, sigma = 5.5, 2.0
w_raw = np.random.normal(loc=mu, scale=sigma, size=I)
w = np.clip(np.round(w_raw), 1, 10).astype(int)
df_prioridad = pd.DataFrame({
    "i": np.arange(1, I + 1),
    "w": w
})
df_prioridad.to_csv(DATA_DIR + "prioridad_pacientes.csv", index=False, sep=SEP)

In [5]:
# Especialidad por cirugía

archivo_especialidades = "S.csv"
archivo_cirugias = "parametros_cirugia.csv"

salida = "especialidad_cirugia.csv"

especialidades = pd.read_csv(archivo_especialidades, sep=";", encoding="utf-8-sig")
cirugias = pd.read_csv(archivo_cirugias, sep=";", encoding="utf-8-sig")

especialidades.columns = especialidades.columns.str.strip()
cirugias.columns = cirugias.columns.str.strip()

especialidades = especialidades.rename(columns={
    "S": "s",
    "Especialidad": "especialidad"
})

cirugias = cirugias.rename(columns={
    "CI": "ci",
    "nombre cirugia": "nombre_cirugia"
})

especialidades["s"] = especialidades["s"].astype(int)
cirugias["ci"] = cirugias["ci"].astype(int)

especialidades["especialidad"] = especialidades["especialidad"].astype(str).str.strip()
cirugias["nombre_cirugia"] = cirugias["nombre_cirugia"].astype(str).str.strip()

especialidades = especialidades.drop_duplicates(subset=["s"])
cirugias = cirugias.drop_duplicates(subset=["ci"])

relacion_ci_s = {
    1: [1],
    2: [1, 2, 3, 15],
    3: [2],
    4: [3],
    5: [4],
    6: [5, 17],
    7: [6],
    8: [7],
    9: [8],
    10: [9],
    11: [10],
    12: [1, 2, 3, 6, 7, 9, 10, 11, 13, 14, 15, 16, 17, 18, 19],
    13: [1, 2, 3, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19],
    14: [1, 2, 3, 6, 8, 9, 11, 12, 15, 16, 17],
    15: [11],
    16: [12],
    17: [13],
    18: [14],
    19: [15],
    20: [15],
    21: [16],
    22: [17],
    23: [18],
    24: [19],
}

relaciones = pd.DataFrame(
    [(ci, s) for ci, lista_s in relacion_ci_s.items() for s in lista_s],
    columns=["ci", "s"]
)
relaciones["valor"] = 1

# Cruce completo ci x s
cirugias["_key"] = 1
especialidades["_key"] = 1

cruce = cirugias.merge(especialidades, on="_key").drop(columns="_key")

cruce = cruce.merge(relaciones, on=["ci", "s"], how="left")
cruce["valor"] = cruce["valor"].fillna(0).astype(int)

# Orden y columnas finales
cruce = cruce.sort_values(["ci", "s"])
cruce = cruce[["ci", "s", "valor"]]

cruce.to_csv(
    salida,
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

print("Archivo generado:")
print(salida)
print("Filas generadas:", len(cruce))

Archivo generado:
especialidad_cirugia.csv
Filas generadas: 456


In [6]:
# Compatibilidad cirujano cirujia

archivo_cirujanos = "K.csv"
archivo_cirugias = "parametros_cirugia.csv"

salida = "cirujano_cirugia.csv"

# Leer archivos
cirujanos = pd.read_csv(archivo_cirujanos, sep=";", encoding="utf-8-sig")
cirugias = pd.read_csv(archivo_cirugias, sep=";", encoding="utf-8-sig")

# Limpiar nombres de columnas
cirujanos.columns = cirujanos.columns.str.strip()
cirugias.columns = cirugias.columns.str.strip()

# Renombrar columnas esperadas
cirujanos = cirujanos.rename(columns={
    "K": "k",
    "Tipo de cirujano": "tipo_cirujano"
})

cirugias = cirugias.rename(columns={
    "CI": "ci",
    "nombre cirugia": "nombre_cirugia"
})

# Tipos correctos
cirujanos["k"] = cirujanos["k"].astype(int)
cirugias["ci"] = cirugias["ci"].astype(int)

# Limpiar textos
cirujanos["tipo_cirujano"] = cirujanos["tipo_cirujano"].astype(str).str.strip()
cirugias["nombre_cirugia"] = cirugias["nombre_cirugia"].astype(str).str.strip()

# Evitar cruces gigantes si hay duplicados
cirujanos = cirujanos.drop_duplicates(subset=["k"])
cirugias = cirugias.drop_duplicates(subset=["ci"])

# Relación cirugía -> cirujanos asociados
# Llave: CI
# Valor: lista de K que pueden realizar esa cirugía
relacion_ci_k = {
    1: [1, 2, 6],             # Cirugía General
    2: [1, 2, 3, 4, 6, 20],   # Cirugía Laparoscópica
    3: [3, 4],                # Cirugía Digestiva
    4: [5, 6],                # Cirugía Ginecológica
    5: [7, 8],                # Cirugía de Urgencia Adulto
    6: [9, 10],               # Cirugía de Urgencia Pediátrica
    7: [11],                  # Cirugía Vascular
    8: [12],                  # Cirugía Proctológica
    9: [13],                  # Cirugía de Tórax
    10: [14],                 # Cirugía de Cabeza y Cuello
    11: [15],                 # Cirugía Máxilo Facial
    12: [1, 2, 3, 4, 5, 6, 9, 10, 11, 12, 14, 15, 18, 19, 20, 21, 22, 23],
    13: [1, 2, 3, 4, 5, 6, 9, 10, 11, 12, 13, 14, 15, 16, 18, 19, 20, 21, 22, 23],
    14: [1, 2, 3, 4, 5, 6, 9, 10, 11, 13, 14, 16, 17, 20, 21, 22],
    15: [16],                 # Cirugía Plástica
    16: [17],                 # Neurocirugía
    17: [18],                 # Cirugía Oftalmológica / LASIK
    18: [19],                 # Otorrinolaringología
    19: [20, 21],             # Urología
    20: [20, 21],             # Litotripsia
    21: [22],                 # Traumatología
    22: [9, 10],              # Cirugía Infantil
    23: [23],                 # Dermatología
    24: [23],                 # Odontología
}

# Tabla de relaciones positivas: valor = 1
relaciones = pd.DataFrame(
    [(ci, k) for ci, lista_k in relacion_ci_k.items() for k in lista_k],
    columns=["ci", "k"]
)
relaciones["valor"] = 1

# Cruce completo CI x K
cirugias["_key"] = 1
cirujanos["_key"] = 1

cruce_detalle = cirugias.merge(cirujanos, on="_key").drop(columns="_key")

# Marcar valor = 1 cuando existe relación; si no, 0
cruce_detalle = cruce_detalle.merge(relaciones, on=["ci", "k"], how="left")
cruce_detalle["valor"] = cruce_detalle["valor"].fillna(0).astype(int)

# Ordenar
cruce_detalle = cruce_detalle.sort_values(["ci", "k"])

# Guardar archivo con detalle
cruce_detalle = cruce_detalle[[
    "ci",
    "nombre_cirugia",
    "k",
    "tipo_cirujano",
    "valor"
]]

cruce_detalle.to_csv(
    salida,
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

# Guardar archivo final ci;k;valor
cruce = cruce_detalle[["ci", "k", "valor"]]

print("Archivos generados:")
print(salida)
print("Filas generadas:", len(cruce))
print("Cruce esperado:", len(cirugias), "cirugías x", len(cirujanos), "cirujanos =", len(cirugias) * len(cirujanos))

Archivos generados:
cirujano_cirugia.csv
Filas generadas: 552
Cruce esperado: 24 cirugías x 23 cirujanos = 552


In [16]:
# Cruce anestesista - cirugía

archivo_cirugias = "parametros_cirugia.csv"
archivo_parametros = "parametros_globales.csv"
salida = "anestesista_cirugia.csv"
cirugias = pd.read_csv(archivo_cirugias, sep=";", encoding="utf-8-sig")
parametros = pd.read_csv(archivo_parametros, sep=";", encoding="utf-8-sig")

cirugias.columns = cirugias.columns.str.strip()
parametros.columns = parametros.columns.str.strip()

cirugias = cirugias.rename(columns={
    "CI": "ci",
    "nombre cirugia": "nombre_cirugia"
})

cirugias["ci"] = cirugias["ci"].astype(int)
cirugias["nombre_cirugia"] = cirugias["nombre_cirugia"].astype(str).str.strip()

# Evitar duplicados
cirugias = cirugias.drop_duplicates(subset=["ci"])

# Obtener cantidad de anestesistas M desde parametros_globales.csv
M = int(parametros.loc[parametros["parametro"] == "M", "valor"].iloc[0])

# Definir cuál anestesista puede asistir cirugía infantil
# Puedes cambiar este valor si quieres que sea otro anestesista
m_infantil = M

# Identificar CI de cirugía infantil usando el nombre
ci_infantil = int(
    cirugias.loc[
        cirugias["nombre_cirugia"].str.lower().str.contains("infantil", na=False),
        "ci"
    ].iloc[0]
)

# Crear listado de anestesistas
anestesistas = pd.DataFrame({
    "m": range(1, M + 1)
})

anestesistas["tipo_anestesista"] = anestesistas["m"].apply(
    lambda x: "Anestesista infantil y general" if x == m_infantil else "Anestesista general"
)

# Cruce completo CI x M
cirugias["_key"] = 1
anestesistas["_key"] = 1

cruce_detalle = cirugias.merge(anestesistas, on="_key").drop(columns="_key")

# Regla:
# - Si la cirugía NO es infantil, todos los anestesistas pueden asistir: valor = 1
# - Si la cirugía ES infantil, solo m_infantil puede asistir: valor = 1
cruce_detalle["valor"] = (
    (cruce_detalle["ci"] != ci_infantil) |
    (
        (cruce_detalle["ci"] == ci_infantil) &
        (cruce_detalle["m"] == m_infantil)
    )
).astype(int)

# Ordenar
cruce_detalle = cruce_detalle.sort_values(["ci", "m"])

# Guardar archivo con detalle
cruce_detalle = cruce_detalle[[
    "ci",
    "nombre_cirugia",
    "m",
    "tipo_anestesista",
    "valor"
]]

cruce_detalle.to_csv(
    salida,
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

cruce = cruce_detalle[["ci", "m", "valor"]]

print("Archivos generados:")
print(salida)
print("Cantidad de anestesistas M:", M)
print("Anestesista habilitado para cirugía infantil:", m_infantil)
print("CI cirugía infantil:", ci_infantil)
print("Filas generadas:", len(cruce))
print("Cruce esperado:", len(cirugias), "cirugías x", M, "anestesistas =", len(cirugias) * M)

Archivos generados:
anestesista_cirugia.csv
Cantidad de anestesistas M: 4
Anestesista habilitado para cirugía infantil: 4
CI cirugía infantil: 22
Filas generadas: 96
Cruce esperado: 24 cirugías x 4 anestesistas = 96


In [11]:
# ============================================================
# 8. COMPATIBILIDAD PABELLÓN-ESPECIALIDAD
# A[p,s] = 1 si pabellón p es compatible con especialidad s
# CSV esperado: p;s;valor
# ============================================================

A = {(p, s): 0 for p in range(1, P + 1) for s in range(1, S + 1)}

for s in range(1, S + 1):
    pabellones_s = np.random.choice(
        range(1, P + 1),
        size=min(2, P),
        replace=False
    )

    for p in pabellones_s:
        A[(p, s)] = 1

# Compatibilidades adicionales
for p in range(1, P + 1):
    for s in range(1, S + 1):
        if np.random.rand() < 0.25:
            A[(p, s)] = 1

df_A = pd.DataFrame({
    "p": [p for p in range(1, P + 1) for s in range(1, S + 1)],
    "s": [s for p in range(1, P + 1) for s in range(1, S + 1)],
    "valor": [A[(p, s)] for p in range(1, P + 1) for s in range(1, S + 1)],
})

df_A.to_csv(DATA_DIR + "pabellon_especialidad.csv", index=False, sep=SEP)


In [ ]:
# ============================================================
# 9. PACIENTE-CIRUGÍA
# G[i,ci] = 1 si paciente i requiere cirugía ci
# CSV esperado: i;ci;valor
#
# REGLA:
# Cada paciente debe tener exactamente una cirugía.
# ============================================================

archivo_cirugias = "parametros_cirugia.csv"
archivo_parametros = "parametros_globales.csv"

salida_detalle = "paciente_cirugia.csv"

# Semilla para replicabilidad
semilla = 123
rng = np.random.default_rng(semilla)

# Leer archivos
cirugias = pd.read_csv(archivo_cirugias, sep=";", encoding="utf-8-sig")
parametros = pd.read_csv(archivo_parametros, sep=";", encoding="utf-8-sig")

# Limpiar nombres de columnas
cirugias.columns = cirugias.columns.str.strip()
parametros.columns = parametros.columns.str.strip()

# Renombrar columnas de cirugía
cirugias = cirugias.rename(columns={
    "CI": "ci",
    "nombre cirugia": "nombre_cirugia"
})

# Tipos correctos
cirugias["ci"] = cirugias["ci"].astype(int)
cirugias["nombre_cirugia"] = cirugias["nombre_cirugia"].astype(str).str.strip()

# Evitar duplicados
cirugias = cirugias.drop_duplicates(subset=["ci"]).sort_values("ci")

# ------------------------------------------------------------
# Obtener cantidad de pacientes I
# ------------------------------------------------------------
# Este bloque intenta leer I desde parametros_globales.csv.
# Debe existir una fila con parametro = I y valor = número de pacientes.
# Si tu archivo usa otro nombre, ajusta "I" abajo.

parametros.columns = parametros.columns.str.lower().str.strip()

if "parametro" in parametros.columns and "valor" in parametros.columns:
    I = int(parametros.loc[parametros["parametro"].astype(str).str.upper() == "I", "valor"].iloc[0])
else:
    raise ValueError("No encontré columnas 'parametro' y 'valor' en parametros_globales.csv")

# ------------------------------------------------------------
# Distribución aproximada de demanda quirúrgica en Chile
# ------------------------------------------------------------
# Los pesos suman 1.
# Se basan en proporciones de lista de espera quirúrgica por especialidad
# y se adaptan a tu clasificación de 24 tipos de cirugía.

pesos_ci = {
    1: 0.035,   # Cirugía General
    2: 0.055,   # Cirugía Laparoscópica
    3: 0.051,   # Cirugía Digestiva
    4: 0.078,   # Cirugía Ginecológica
    5: 0.025,   # Cirugía de Urgencia Adulto
    6: 0.008,   # Cirugía de Urgencia Pediátrica
    7: 0.012,   # Cirugía Vascular
    8: 0.012,   # Cirugía Proctológica
    9: 0.005,   # Cirugía de Tórax
    10: 0.008,  # Cirugía de Cabeza y Cuello
    11: 0.010,  # Cirugía Máxilo Facial
    12: 0.025,  # Cirugía Electiva de baja complejidad
    13: 0.025,  # Cirugía Electiva de mediana complejidad
    14: 0.010,  # Cirugía Electiva de alta complejidad
    15: 0.012,  # Cirugía Plástica
    16: 0.010,  # Neurocirugía
    17: 0.069,  # Cirugía Oftalmológica / LASIK
    18: 0.072,  # Otorrinolaringología
    19: 0.073,  # Urología
    20: 0.015,  # Litotripsia
    21: 0.224,  # Traumatología
    22: 0.012,  # Cirugía Infantil
    23: 0.129,  # Dermatología
    24: 0.025,  # Odontología
}

# Validar que todos los CI del archivo tengan peso
ci_archivo = set(cirugias["ci"])
ci_pesos = set(pesos_ci.keys())

faltan_pesos = ci_archivo - ci_pesos
pesos_sobrantes = ci_pesos - ci_archivo

if faltan_pesos:
    raise ValueError(f"Faltan pesos para estos CI: {sorted(faltan_pesos)}")

if pesos_sobrantes:
    print(f"Advertencia: hay pesos para CI que no están en el archivo: {sorted(pesos_sobrantes)}")

# Crear vector de probabilidades en el mismo orden que cirugias
cirugias["peso"] = cirugias["ci"].map(pesos_ci)

# Normalizar por seguridad
cirugias["probabilidad"] = cirugias["peso"] / cirugias["peso"].sum()

# ------------------------------------------------------------
# Asignar exactamente una cirugía a cada paciente
# ------------------------------------------------------------

pacientes = pd.DataFrame({
    "i": range(1, I + 1)
})

asignaciones = pd.DataFrame({
    "i": pacientes["i"],
    "ci_asignado": rng.choice(
        cirugias["ci"].to_numpy(),
        size=I,
        replace=True,
        p=cirugias["probabilidad"].to_numpy()
    )
})

# ------------------------------------------------------------
# Crear matriz completa paciente x cirugía
# ------------------------------------------------------------

pacientes["_key"] = 1
cirugias_aux = cirugias[["ci", "nombre_cirugia", "probabilidad"]].copy()
cirugias_aux["_key"] = 1

cruce = pacientes.merge(cirugias_aux, on="_key").drop(columns="_key")

cruce = cruce.merge(asignaciones, on="i", how="left")

cruce["valor"] = (cruce["ci"] == cruce["ci_asignado"]).astype(int)

# ------------------------------------------------------------
# Validación: cada paciente debe tener exactamente una cirugía
# ------------------------------------------------------------

validacion = cruce.groupby("i")["valor"].sum()

if not (validacion == 1).all():
    raise ValueError("Error: hay pacientes con cero o más de una cirugía asignada.")

# ------------------------------------------------------------
# Guardar detalle
# ------------------------------------------------------------

cruce_detalle = cruce[[
    "i",
    "ci",
    "nombre_cirugia",
    "probabilidad",
    "valor"
]].sort_values(["i", "ci"])

cruce_detalle.to_csv(
    salida_detalle,
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# Guardar archivo final i;ci;valor
# ------------------------------------------------------------

cruce_final = cruce_detalle[["i", "ci", "valor"]]

cruce_final.to_csv(
    salida,
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# Resumen de asignaciones generadas
# ------------------------------------------------------------

resumen = (
    asignaciones
    .merge(cirugias[["ci", "nombre_cirugia", "probabilidad"]], left_on="ci_asignado", right_on="ci")
    .groupby(["ci", "nombre_cirugia", "probabilidad"])
    .size()
    .reset_index(name="pacientes_asignados")
)

resumen["porcentaje_simulado"] = resumen["pacientes_asignados"] / I

resumen = resumen.sort_values("ci")

print("Archivos generados:")
print(salida_detalle)
print()
print("Cantidad de pacientes:", I)
print("Cantidad de cirugías:", len(cirugias))
print("Filas generadas:", len(cruce_final))
print("Cruce esperado:", I, "pacientes x", len(cirugias), "cirugías =", I * len(cirugias))
print()
print("Validación: cada paciente tiene exactamente una cirugía.")
print()
print(resumen)


Archivos generados:
paciente_cirugia_detalle.csv

Cantidad de pacientes: 1500
Cantidad de cirugías: 24
Filas generadas: 36000
Cruce esperado: 1500 pacientes x 24 cirugías = 36000

Validación: cada paciente tiene exactamente una cirugía.

    ci                            nombre_cirugia  probabilidad  \
0    1                           Cirugía General         0.035   
1    2                     Cirugía Laparoscópica         0.055   
2    3                         Cirugía Digestiva         0.051   
3    4                      Cirugía Ginecológica         0.078   
4    5                Cirugía de Urgencia Adulto         0.025   
5    6            Cirugía de Urgencia Pediátrica         0.008   
6    7                          Cirugía Vascular         0.012   
7    8                      Cirugía Proctológica         0.012   
8    9                          Cirugía de Tórax         0.005   
9   10                Cirugía de Cabeza y Cuello         0.008   
10  11                     Cirugía M

In [13]:
# ============================================================
# 10. DISPONIBILIDAD DE PABELLONES
# DispP[p,t] = 1 si pabellón p está disponible en tiempo t
# CSV esperado: p;t;valor
#
# REGLA:
# Todos los pabellones están libres en todos los tiempos.
# ============================================================

df_DispP = pd.DataFrame({
    "p": [p for p in range(1, P + 1) for t in range(1, T + 1)],
    "t": [t for p in range(1, P + 1) for t in range(1, T + 1)],
    "valor": 1,
})

df_DispP.to_csv(
    DATA_DIR + "disponibilidad_pabellones.csv",
    index=False,
    sep=SEP
)

In [14]:
# ============================================================
# 11. DISPONIBILIDAD DE CIRUJANOS
# DispK[k,t] = 1 si cirujano k está disponible en tiempo t
# CSV esperado: k;t;valor
#
# REGLA:
# Cada cirujano atiende hasta 8 bloques seguidos.
# Luego descansa hasta 2 bloques seguidos.
# ============================================================

rng = np.random.default_rng(123)

DispK = {}

for k in range(1, K + 1):

    t = 1

    # Para que no todos los cirujanos tengan exactamente el mismo patrón,
    # algunos parten disponibles y otros descansando.
    estado = 1 if rng.random() < 0.80 else 0

    while t <= T:

        if estado == 1:
            # Atiende entre 4 y 8 bloques seguidos, máximo 8
            duracion = rng.integers(4, 9)
        else:
            # Descansa entre 1 y 2 bloques seguidos, máximo 2
            duracion = rng.integers(1, 3)

        for _ in range(duracion):
            if t > T:
                break

            DispK[(k, t)] = estado
            t += 1

        # Alterna entre atender y descansar
        estado = 1 - estado

df_DispK = pd.DataFrame({
    "k": [k for k in range(1, K + 1) for t in range(1, T + 1)],
    "t": [t for k in range(1, K + 1) for t in range(1, T + 1)],
    "valor": [DispK[(k, t)] for k in range(1, K + 1) for t in range(1, T + 1)],
})

df_DispK.to_csv(
    DATA_DIR + "disponibilidad_cirujanos.csv",
    index=False,
    sep=SEP
)

In [15]:
# ============================================================
# 12. DISPONIBILIDAD DE ANESTESISTAS
# DispM[m,t] = 1 si anestesista m está disponible en tiempo t
# CSV esperado: m;t;valor
#
# REGLA:
# Cada anestesista atiende hasta 8 bloques seguidos.
# Luego descansa hasta 2 bloques seguidos.
# ============================================================

rng = np.random.default_rng(123)

DispM = {}

for m in range(1, M + 1):

    t = 1

    # Para que no todos los anestesistas tengan exactamente el mismo patrón
    estado = 1 if rng.random() < 0.80 else 0

    while t <= T:

        if estado == 1:
            # Atiende entre 4 y 8 bloques seguidos, máximo 8
            duracion = rng.integers(4, 9)
        else:
            # Descansa entre 1 y 2 bloques seguidos, máximo 2
            duracion = rng.integers(1, 3)

        for _ in range(duracion):
            if t > T:
                break

            DispM[(m, t)] = estado
            t += 1

        # Alterna entre atender y descansar
        estado = 1 - estado

df_DispM = pd.DataFrame({
    "m": [m for m in range(1, M + 1) for t in range(1, T + 1)],
    "t": [t for m in range(1, M + 1) for t in range(1, T + 1)],
    "valor": [DispM[(m, t)] for m in range(1, M + 1) for t in range(1, T + 1)],
})

df_DispM.to_csv(
    DATA_DIR + "disponibilidad_anestesistas.csv",
    index=False,
    sep=SEP
)